# Workshop 3.1: From Single Stock to a Universe

Welcome to Stage 3! In Stage 2, we learned how to download market data, measure volatility, and backtest a single-asset trend strategy on Apple. While single-asset testing builds intuition, institutional quantitative funds rarely trade one stock in isolation.

### What is a Quantitative Universe?

In professional quant management, the collection of assets evaluated and traded by a strategy is called an **investment universe**. A universe might encompass all large-cap equities in an index, or focus on a curated basket of technology leaders.

Managing multiple assets presents an engineering challenge: how should we organize dozens of stocks with overlapping dates inside pandas? Creating separate variables for every stock leads to messy code and slow loops.

The institutional solution is to stack all stocks into a single DataFrame indexed by both **Date** and **Ticker**. This two-dimensional index is known as a **MultiIndex**.

In this workshop, we will fetch multiple assets, structure them into a sorted MultiIndex, and build a reusable universe generator function.

> **Key Takeaway**: An investment universe bundles multiple assets into a unified MultiIndex DataFrame, allowing our models to process portfolios simultaneously.

## Topic 1: Fetching Multiple Stocks Individually

To understand multi-asset mechanics from the ground up, let's start by downloading two distinct stocks: Apple (`AAPL`) and Microsoft (`MSFT`).

Let's retrieve daily price histories for both companies across 2023. Let's see:

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# Define tickers and date range for 2023:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
start = "2023-01-01"
end = "2024-01-01"

# Fetch AAPL:
aapl = yf.download("AAPL", start=start, end=end, progress=False)
if isinstance(aapl.columns, pd.MultiIndex):
    aapl.columns = aapl.columns.get_level_values(0)
aapl.index = aapl.index.tz_localize(None)

# Fetch MSFT:
msft = yf.download("MSFT", start=start, end=end, progress=False)
if isinstance(msft.columns, pd.MultiIndex):
    msft.columns = msft.columns.get_level_values(0)
msft.index = msft.index.tz_localize(None)

print(f"AAPL rows: {len(aapl)}, MSFT rows: {len(msft)}")
print("AAPL columns:", aapl.columns.tolist())

Downloaded AAPL shape: (250, 5)
Downloaded MSFT shape: (250, 5)


> **Key Takeaway**: We begin by downloading individual asset price histories and standardizing their column structures.

---

## Topic 2: Tagging Each DataFrame with a Ticker Column

Before stacking our tables together, we must label which rows belong to which stock. Without an explicit ticker column, combining the tables would create an anonymous soup of prices where we cannot distinguish Apple from Microsoft.

We insert a constant `"Ticker"` column into each table before merging.

Let's tag our tables and inspect the additions. Let's check:

In [2]:
# Add Ticker column to each DataFrame:
aapl["Ticker"] = "AAPL"
msft["Ticker"] = "MSFT"

print("AAPL with Ticker column:")
print(aapl[["Close", "Ticker"]].head(3))

print("\nMSFT with Ticker column:")
print(msft[["Close", "Ticker"]].head(3))

AAPL with Ticker column (first 3 rows):
                 Close Ticker
Date                         
2023-01-03  122.876732   AAPL
2023-01-04  124.144127   AAPL
2023-01-05  122.827614   AAPL

MSFT with Ticker column (first 3 rows):
                 Close Ticker
Date                         
2023-01-03  232.510559   MSFT
2023-01-04  222.339813   MSFT
2023-01-05  219.626465   MSFT


> **Key Takeaway**: Tagging individual DataFrames with a `Ticker` column ensures asset identity is preserved when combining data.

---

## Topic 3: Concatenating DataFrames with pd.concat()

Now that both tables share identical column schemas and carry explicit ticker labels, we can stack them vertically using **`pd.concat()`**.

Passing our list of DataFrames into `pd.concat([aapl, msft])` stitches the rows together, doubling our total observation count.

Let's concatenate the tables and verify the combined dimensions. Let's see:

In [3]:
# Stack the DataFrames vertically:
combined = pd.concat([aapl, msft])

print("Combined head (first 5 rows from Apple):")
print(combined[["Close", "Ticker"]].head())

print("\nCombined tail (last 5 rows from Microsoft):")
print(combined[["Close", "Ticker"]].tail())

print(f"\nCombined shape: {combined.shape}")

Combined head (first 5 rows from Apple):
                 Close Ticker
Date                         
2023-01-03  122.876732   AAPL
2023-01-04  124.144127   AAPL
2023-01-05  122.827614   AAPL
2023-01-06  127.346939   AAPL
2023-01-09  127.867622   AAPL

Combined tail (last 5 rows from Microsoft):
                 Close Ticker
Date                         
2023-12-22  372.417389   MSFT
2023-12-26  372.337799   MSFT
2023-12-27  371.770935   MSFT
2023-12-28  373.083679   MSFT
2023-12-29  373.829559   MSFT

Combined shape: (500, 6)


> **Key Takeaway**: `pd.concat()` stacks tagged DataFrames into a single continuous table.

---

## Topic 4: Constructing a MultiIndex

In our concatenated table, the row index currently consists solely of `Date`. Because both Apple and Microsoft traded on the same trading dates, every calendar date appears twice in our index. Duplicate dates make filtering ambiguous and slow.

The institutional solution is a two-level **MultiIndex**: `["Date", "Ticker"]`.

We build this hierarchy in three clear steps:
- Move `Date` from the index into a standard column with `.reset_index()`.
- Set both `["Date", "Ticker"]` as our composite index using `.set_index()`.
- Sort the index hierarchically with `.sort_index()`, which optimizes fast lookups and date slicing.

Let's transform our table into a sorted MultiIndex. Let's see:

In [4]:
# First, we move Date from index to column:
combined_reset = combined.reset_index()

# Next, we set our two-level index [Date, Ticker]:
combined_multi = combined_reset.set_index(["Date", "Ticker"])

# Finally, we sort the MultiIndex for fast lookups:
combined_multi = combined_multi.sort_index()

# Print result to show the MultiIndex:
print("MultiIndex DataFrame head (first 6 rows):")
print(combined_multi.head(6))

MultiIndex DataFrame head (first 6 rows):
                           Close        High         Low        Open     Volume
Date       Ticker                                                          
2023-01-03 AAPL      122.876732  128.604489  121.992513  127.995367  112117500
           MSFT      232.510559  238.498495  230.394878  235.907282   25740000
2023-01-04 AAPL      124.144127  126.403797  122.886574  124.664832   89113600
           MSFT      222.339813  225.998559  219.292468  225.425972   50623400
2023-01-05 AAPL      122.827614  125.529389  122.572179  124.900614   80962700
           MSFT      219.626465  220.590161  214.937748  219.864871   39585600


> **Key Takeaway**: A two-level MultiIndex indexed by `["Date", "Ticker"]` provides unique coordinate addresses for every asset on every date.

---

## Topic 5: Querying a MultiIndex with .loc and .xs()

Navigating multi-level indices can feel unfamiliar at first, but pandas provides intuitive tools for slicing across different dimensions:
- `combined_multi.loc["2023-06-01"]`: Extracts a cross-sectional snapshot of all stocks on a specific trading day.
- `combined_multi.xs("AAPL", level="Ticker")`: Extracts an asset-specific slice for Apple across the entire backtest.

The name `.xs()` stands for *cross-section*. It acts like an MRI scan, cleanly slicing out one specific level of our MultiIndex while preserving the remaining index structure.

Let's test both cross-sectional and asset-specific queries. Let's check:

In [5]:
# Inspect index names:
print("Index names:")
print(combined_multi.index.names)

# Inspect index levels:
print("\nIndex levels count:")
print(f"Level 0 (Dates): {len(combined_multi.index.levels[0])} unique dates")
print(f"Level 1 (Tickers): {combined_multi.index.levels[1].tolist()}")

# Select all stocks on a single date using .loc:
print("\nAll stocks on 2023-06-01 (loc):")
print(combined_multi.loc["2023-06-01"])

# Select all dates for AAPL using .xs():
print("\nAll dates for AAPL (xs) first 5 rows:")
print(combined_multi.xs("AAPL", level="Ticker").head())

Index names:
['Date', 'Ticker']

Index levels count:
Level 0 (Dates): 250 unique dates
Level 1 (Tickers): ['AAPL', 'MSFT']

All stocks on 2023-06-01 (loc):
             Close        High         Low        Open    Volume
Ticker                                                          
AAPL    177.447159  177.476717  174.333529  175.092233  68901800
MSFT    324.282104  325.208414  316.618225  317.798028  26773900

All dates for AAPL (xs) first 5 rows:
                 Close        High         Low        Open     Volume
Date                                                                 
2023-01-03  122.876732  128.604489  121.992513  127.995367  112117500
2023-01-04  124.144127  126.403797  122.886574  124.664832   89113600
2023-01-05  122.827614  125.529389  122.572179  124.900614   80962700
2023-01-06  127.346939  128.005188  122.699890  123.800253   87754700
2023-01-09  127.867622  131.070463  127.612187  128.182018   70790800


> **Key Takeaway**: Use `.loc` to slice dates on level 0, and `.xs()` to isolate specific assets across level 1.

---

## Topic 6: Building a Reusable Universe Generator

Manually downloading and reindexing stocks becomes tedious when managing large portfolios. We can encapsulate this entire pipeline inside a reusable helper function: `build_universe()`.

Our helper accepts a list of ticker symbols alongside start and end dates, downloads each asset, and returns a fully sorted MultiIndex DataFrame.

Let's test our helper across all five mega-cap tech stocks. Let's see:

In [6]:
# Define universe builder function:
def build_universe(ticker_list, start_date, end_date):
    frames = []
    for ticker in ticker_list:
        df = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df["Ticker"] = ticker
        frames.append(df)
    combined = pd.concat(frames)
    combined = combined.reset_index().set_index(["Date", "Ticker"]).sort_index()
    return combined

# Test the helper function with all 5 tickers:
universe = build_universe(tickers, start, end)

# Print shape and first 10 rows:
print(f"Universe shape: {universe.shape}")
print(f"Index names: {universe.index.names}")
print("\nFirst 10 rows of the 5-stock universe:")
print(universe.head(10))

Universe shape: (1250, 5)
Index names: ['Date', 'Ticker']

First 10 rows of the 5-stock universe:
                           Close        High         Low        Open     Volume
Date       Ticker                                                          
2023-01-03 AAPL      122.876732  128.604489  121.992513  127.995367  112117500
           AMZN       85.820000   86.959999   84.209999   85.459999   76706000
           GOOGL      88.336708   90.249745   87.741976   88.802571   28131200
           MSFT      232.510559  238.498495  230.394878  235.907282   25740000
           NVDA       14.283262   14.962753   14.064748   14.818074  401277000
2023-01-04 AAPL      124.144127  126.403797  122.886574  124.664832   89113600
           AMZN       85.139999   86.980003   83.360001   86.550003   68885100
           GOOGL      87.305840   89.853251   86.502954   89.555884   34854800
           MSFT      222.339813  225.998559  219.292468  225.425972   50623400
           NVDA       14.716300   1

> **Key Takeaway**: Encapsulating data extraction into a `build_universe()` function provides a clean, repeatable foundation for multi-asset research.

---

## Practice Time

Now it is your turn to construct and query multi-asset universes. Getting comfortable with multi-level indices takes practice, but it is one of the most powerful tools in quantitative engineering.

---

### Challenge 1: Constructing a Three-Asset Tech Universe

- Build a universe containing three stocks (`["TSLA", "META", "NFLX"]`) across the 2023 calendar year using `build_universe()`.
- Display the overall table dimensions and verify the unique ticker symbols in the index.

In [ ]:
# Challenge 1: Build a three-stock universe for 2023
# Write your code below this line:




### Challenge 2: Cross-Sectional and Asset Slicing with .xs()

- From your 3-stock universe created in Challenge 1:
- Use `.xs()` to isolate all historical price records for `TSLA` and display the resulting shape.
- Use `.xs()` to extract a cross-sectional snapshot of all three stocks on `2023-06-01`.

In [ ]:
# Challenge 2: Use xs() to extract TSLA data and 2023-06-01 data
# Write your code below this line:




### Challenge 3: Counting Unique Universe Tickers

- Write a Python function named `count_unique_tickers(df)` that accepts any MultiIndex DataFrame with a `"Ticker"` level and returns the count of unique tickers.
- Test your function on both `universe` and your Challenge 1 universe.

In [ ]:
# Challenge 3: Write and test count_unique_tickers(df)
# Write your code below this line:




---

## Solutions Section

Terrific work completing these universe construction challenges! Mastering MultiIndex tables enables you to manage complex multi-asset portfolios cleanly.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
my_tickers = ["TSLA", "META", "NFLX"]
ex1_universe = build_universe(my_tickers, "2023-01-01", "2024-01-01")

print(f"Exercise 1 Universe Shape: {ex1_universe.shape}")
print(f"Unique Tickers: {ex1_universe.index.get_level_values('Ticker').unique().tolist()}")
```

#### Solution for Challenge 2:
```python
# 1. Extract all TSLA data using .xs():
tsla_df = ex1_universe.xs("TSLA", level="Ticker")
print(f"TSLA DataFrame Shape: {tsla_df.shape}")

# 2. Extract cross-section on 2023-06-01:
date_df = ex1_universe.xs("2023-06-01", level="Date")
print("\nCross-section on 2023-06-01:")
print(date_df)
```

#### Solution for Challenge 3:
```python
def count_unique_tickers(df):
    return df.index.get_level_values("Ticker").nunique()

print(f"Tickers in 5-stock universe: {count_unique_tickers(universe)}")
print(f"Tickers in 3-stock universe: {count_unique_tickers(ex1_universe)}")
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [7]:
# Solution for Challenge 1:
my_tickers = ["TSLA", "META", "NFLX"]
ex1_universe = build_universe(my_tickers, "2023-01-01", "2024-01-01")

print(f"Exercise 1 Universe Shape: {ex1_universe.shape}")
print(f"Unique Tickers: {ex1_universe.index.get_level_values('Ticker').unique().tolist()}")

Exercise 1 Universe Shape: (750, 5)
Unique Tickers: ['META', 'NFLX', 'TSLA']


In [8]:
# Solution for Challenge 2:
# We extract all TSLA data using .xs():
tsla_df = ex1_universe.xs("TSLA", level="Ticker")
print(f"TSLA DataFrame Shape: {tsla_df.shape}")

# We extract the cross-section on 2023-06-01:
date_df = ex1_universe.xs("2023-06-01", level="Date")
print("\nCross-section on 2023-06-01:")
print(date_df)

TSLA DataFrame Shape: (250, 5)

Cross-section on 2023-06-01:
             Close        High         Low        Open     Volume
Ticker                                                           
META    270.236908  271.614822  263.575435  263.585327   25609500
NFLX     40.313000   40.751999   39.307999   39.741001   71601000
TSLA    207.520004  209.800003  199.369995  202.589996  148029900


In [9]:
# Solution for Challenge 3:
def count_unique_tickers(df):
    return df.index.get_level_values("Ticker").nunique()

print(f"Tickers in 5-stock universe: {count_unique_tickers(universe)}")
print(f"Tickers in 3-stock universe: {count_unique_tickers(ex1_universe)}")

Tickers in 5-stock universe: 5
Tickers in 3-stock universe: 3
